# Gemma4NPC - Base Training

This notebook trains the base `google/gemma-4-12B-it` model on the **PIPPA** roleplay dataset (converted to Gemma 4 native chat format).

## Environment
- Recommended: A100 (40GB or 80GB) or equivalent
- Uses Unsloth for fast, memory-efficient LoRA training
- Critical: We use Gemma 4 native delimiters (`<|turn>user\n`, `<|turn>model\n`)

In [ ]:
!pip install "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch

max_seq_length = 4096 # Fits 99% of PIPPA conversations

model, tokenizer = FastModel.from_pretrained(
    model_name = "google/gemma-4-12B-it",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Add LoRA adapters
model = FastModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

In [ ]:
# Load Gemma 4 Chat Template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

from datasets import load_dataset
dataset = load_dataset("json", data_files="data/final/pippa_gemma4.jsonl", split="train")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat, num_proc=4)

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import math

# NaN Protection Callback
class NaNDetectionCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            loss = logs["loss"]
            if math.isnan(loss) or math.isinf(loss):
                print(f"\n🚨 NaN loss detected at step {state.global_step}! Stopping.\n")
                control.should_training_stop = True

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        output_dir = "outputs/Gemma4NPC",
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        num_train_epochs = 1,
        learning_rate = 2e-5,
        lr_scheduler_type = "cosine",
        warmup_steps = 800,
        max_grad_norm = 0.4, # Critical for Gemma 4 stability
        weight_decay = 0.01,
        optim = "adamw_8bit",
        bf16 = True,
        logging_steps = 10,
        save_steps = 500,
        seed = 42,
        report_to = "none",
    ),
)

# Mask user input - ONLY train on model responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)
trainer.add_callback(NaNDetectionCallback())

In [ ]:
# Start training
trainer_stats = trainer.train()

In [ ]:
model.save_pretrained_merged("outputs/Gemma4NPC/merged_float16", tokenizer, save_method="merged_16bit")